In [1]:
import json

In [2]:
with open("./metadata.json", "r") as f:
    metadata = json.load(f)

In [3]:
with open("./data.json", "r") as f:
    data = json.load(f)

In [4]:
for node in data["nodes"]:
    if "Turing" in node["title"]:
        print(node["raw_text"])
        break

As well as applying differencing to the full 5-bit characters of the ITA2 code, it was also applied to the individual impulses (bits). So, for the first impulse, that was enciphered by wheels χ 1 {\displaystyle \chi _{1}} and ψ 1 {\displaystyle \psi _{1}} , differenced at one: Δ K 1 = K 1 ⊕ K 1 _ {\displaystyle \Delta K_{1}=K_{1}\oplus {\underline {K_{1}}}} And for the second impulse: Δ K 2 = K 2 ⊕ K 2 _ {\displaystyle \Delta K_{2}=K_{2}\oplus {\underline {K_{2}}}} And so on. It is also worth noting that the periodicity of the chi and psi wheels for each impulse (41 and 43 respectively for the first one) is reflected in its pattern of Δ K {\displaystyle \Delta K} . However, given that the psi wheels did not advance for every input character, as did the chi wheels, it was not simply a repetition of the pattern every 41 × 43 = 1763 characters for Δ K 1 {\displaystyle \Delta K_{1}} , but a more complex sequence.

Cryptanalysis often involves finding patterns of some sort that provide a wa

In [5]:
import pandas as pd
import json
import os

def load_and_group_csv(csv_path):
    print(f"Loading {csv_path}... (this might take a moment)")
    df = pd.read_csv(csv_path, usecols=['page_title', 'text'])
    
    df['page_title'] = df['page_title'].str.replace(' ', '_').str.strip()
    
    print("grouping sections into full articles...")
    df_grouped = df.groupby('page_title')['text'].apply(lambda x: '\n\n'.join(x.astype(str))).reset_index()
    
    lookup_dict = pd.Series(df_grouped.text.values, index=df_grouped.page_title).to_dict()
    
    print(f"Loaded {len(lookup_dict)} unique articles.")
    return lookup_dict

def save_json_safe(data, filepath):
    temp_path = filepath + ".tmp"
    with open(temp_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4)
    if os.path.exists(filepath):
        os.remove(filepath)
    os.rename(temp_path, filepath)
    print(f"--> Checkpoint saved to {filepath}")

In [6]:
METADATA_FILE = 'data.json'
CSV_FILE = 'wiki_stem_corpus.csv'
SAVE_EVERY_N = 100

if os.path.exists(METADATA_FILE):
    with open(METADATA_FILE, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    print("Loaded existing metadata.json")
else:
    print("ERROR: metadata.json not found!")

if 'text_lookup' not in locals():
    text_lookup = load_and_group_csv(CSV_FILE)

nodes = metadata.get('nodes', [])
updates_count = 0
matches_found = 0

print(f"Processing {len(nodes)} nodes...")

for i, node in enumerate(nodes):
    if 'raw_text' in node and node['raw_text']:
        continue

    title = node.get('title', '')
    
    text = text_lookup.get(title)
    
    if not text:
         text = text_lookup.get(title.capitalize())
    
    if text:
        node['raw_text'] = text
        matches_found += 1
    else:
        node['raw_text'] = None 
    
    updates_count += 1

    if updates_count > 0 and updates_count % SAVE_EVERY_N == 0:
        print(f"Processed {i+1}/{len(nodes)} nodes. Found {matches_found} texts so far.")
        save_json_safe(metadata, METADATA_FILE)

save_json_safe(metadata, METADATA_FILE)
print(f"DONE. Total texts matched: {matches_found}")

Loaded existing metadata.json
Loading wiki_stem_corpus.csv... (this might take a moment)
grouping sections into full articles...
Loaded 404209 unique articles.
Processing 11701 nodes...
--> Checkpoint saved to data.json
DONE. Total texts matched: 0


In [7]:
missing_titles = [node['title'] for node in nodes if node.get('raw_text') is None]
print(f"Titles with missing text: {len(missing_titles)}")

Titles with missing text: 0


In [8]:
import wikipediaapi
import json
import time
import os

SAVE_FILE = 'data.json'
USER_AGENT = 'WikiCS_Rehydrator/1.0 (contact: uregilsahin@gmail.com)'

with open(SAVE_FILE, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

nodes = metadata.get('nodes', [])

wiki_wiki = wikipediaapi.Wikipedia(
    user_agent=USER_AGENT,
    language='en',
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

missing_nodes = [n for n in nodes if not n.get('raw_text')]
print(f"Starting repair on {len(missing_nodes)} missing nodes...")

updates_count = 0
start_time = time.time()

for i, node in enumerate(missing_nodes):
    title = node['title']
    
    clean_title = title.replace('_', ' ')
    
    try:
        page = wiki_wiki.page(clean_title)
        
        if page.exists():
            node['raw_text'] = page.text
            print(f"[✓] Found: {title} (as '{page.title}')")
        else:
            print(f"[X] Dead Link: {title}")
            node['raw_text'] = "<PAGE_NOT_FOUND>"
            
    except Exception as e:
        print(f"[!] Error on {title}: {e}")
    
    updates_count += 1
    
    if updates_count % 50 == 0:
        with open(SAVE_FILE + '.tmp', 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=4)
        if os.path.exists(SAVE_FILE):
             os.remove(SAVE_FILE)
        os.rename(SAVE_FILE + '.tmp', SAVE_FILE)
        print(f"--> Saved progress at {updates_count}/{len(missing_nodes)}")

    time.sleep(1.0) 


with open(SAVE_FILE, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=4)

print("Repair job complete.")

Starting repair on 0 missing nodes...
Repair job complete.
